# Milestone 6 - Part 1: RAG Pipeline

This notebook implements a complete RAG workflow:
**documents -> chunking -> embeddings -> FAISS index -> retrieval -> local LLM generation**.

For final submission:
- Use a real **open-weight 7B-14B instruct model**.
- Run inference in a **local or self-hosted** environment.
- Use an **open-source vector database** (FAISS is used here).


## Step 0: Expected Outputs

By the end of this notebook:
1. Build a reproducible FAISS index from `data/corpus/`.
2. Run `retrieve(query, k)` and return top-k chunks with scores and sources.
3. Run grounded answer generation using retrieved context and Ollama.
4. Compute a simple retrieval metric from `data/eval_queries.json`.


## Step 1: Paths and Hyperparameters

- `CHUNK_SIZE` / `CHUNK_OVERLAP`: controls chunk granularity and context continuity.
- `TOP_K`: number of retrieved passages used for evidence.
- `EMBED_MODEL_NAME`: embedding model for retrieval.
- `OLLAMA_MODEL`: generation model for grounded answers.

These are key design choices to justify in the report.


In [20]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from pathlib import Path

# Project root directory
ROOT = Path(".").resolve()
CORPUS_DIR = ROOT / "data" / "corpus"
EVAL_PATH = ROOT / "data" / "eval_queries.json"

# Chunking hyperparameters
CHUNK_SIZE = 450
CHUNK_OVERLAP = 90

# Retrieval hyperparameter
TOP_K = 4

# Embedding model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# LLM (Ollama)
# Example setup: ollama pull mistral:7b-instruct
OLLAMA_MODEL = "mistral:7b-instruct"
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

print("ROOT =", ROOT)
print("CORPUS_DIR exists:", CORPUS_DIR.exists(), CORPUS_DIR)


ROOT = /Users/YUNA/Desktop/ids568-milestone6-660737046
CORPUS_DIR exists: True /Users/YUNA/Desktop/ids568-milestone6-660737046/data/corpus


## Step 2: Document Ingestion

Load all `.txt`/`.md` files from `data/corpus/` while preserving source filenames.
Source tracking is required for grounded citations in final answers.


In [ ]:
def load_corpus_texts(corpus_dir: Path) -> list[dict]:
    paths = sorted(set(corpus_dir.glob("*.txt")) | set(corpus_dir.glob("*.md")))
    print(f"glob: {len(paths)} files under {corpus_dir}")
    docs: list[dict] = []
    for p in paths:
        t0 = time.perf_counter()
        text = p.read_text(encoding="utf-8")
        dt = time.perf_counter() - t0
        if dt > 0.5:
            print(f"  slow read {dt:.2f}s -> {p.name} (iCloud/Desktop? see Step 2 markdown)")
        docs.append({"source": p.name, "text": text.strip() + "\n"})
    if not docs:
        raise FileNotFoundError(f"No .txt or .md files found under: {corpus_dir}")
    return docs

raw_docs = load_corpus_texts(CORPUS_DIR)
print(f"Loaded {len(raw_docs)} documents")
for d in raw_docs:
    print("-", d["source"], "chars=", len(d["text"]))


## Step 3: Chunking

This notebook uses fixed-size character chunking with overlap.

Report rationale should explain:
- why chunk size is appropriate,
- why overlap is needed,
- why the embedding model matches the corpus domain/language.


In [15]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks: list[str] = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(text[start:end])
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks


def build_chunks(raw_docs: list[dict], chunk_size: int, overlap: int) -> list[dict]:
    rows: list[dict] = []
    for doc in raw_docs:
        parts = chunk_text(doc["text"], chunk_size, overlap)
        for i, c in enumerate(parts):
            c = c.strip()
            if not c:
                continue
            rows.append(
                {
                    "chunk_id": f"{doc['source']}::{i}",
                    "source": doc["source"],
                    "chunk_index": i,
                    "text": c,
                }
            )
    return rows

chunks = build_chunks(raw_docs, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Total chunks: {len(chunks)}")
print("Example chunk:\n---\n", chunks[0]["text"][:400], "\n---")


Total chunks: 1032
Example chunk:
---
 Wikipedia (English)
Article title: Big data
Page URL: https://en.wikipedia.org/wiki/Big_data
License: CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/)
Retrieved (UTC): 2026-04-20
Text is from Wikipedia parse API (HTML stripped in-script; tables/figures may be lossy).

--- Article text ---

Extremely large or complex datasets

 Not to be confused with Big Data (band) .

 A diagram of  
---


## Step 4: Embeddings and FAISS Index

Pipeline:
1. Encode chunks with `SentenceTransformer`.
2. L2-normalize embeddings.
3. Build `faiss.IndexFlatIP` and add all vectors.

With normalized vectors, inner product ranking is cosine-equivalent.


In [16]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in chunks]

t0 = time.perf_counter()
emb = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
embed_wall_s = time.perf_counter() - t0

emb = emb.astype("float32")
dim = emb.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(emb)

print("embedding shape:", emb.shape)
print("index.ntotal:", index.ntotal)
print(f"wall time for encoding all chunks: {embed_wall_s:.3f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

embedding shape: (1032, 384)
index.ntotal: 1032
wall time for encoding all chunks: 3.908s


## Step 5: Retrieval Function

For each query:
1. embed and normalize query vector,
2. run `index.search` for top-k results,
3. return chunk text, source, and score.

Retrieval latency here includes query encoding + FAISS search.


In [17]:
def retrieve(query: str, k: int) -> tuple[list[dict], float]:
    t0 = time.perf_counter()
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    dt = time.perf_counter() - t0

    hits: list[dict] = []
    for score, i in zip(scores[0].tolist(), idxs[0].tolist()):
        if i < 0:
            continue
        c = chunks[i]
        hits.append(
            {
                "chunk_id": c["chunk_id"],
                "source": c["source"],
                "score": float(score),
                "text": c["text"],
            }
        )
    return hits, dt


demo_hits, demo_dt = retrieve("What is RAG?", TOP_K)
print(f"retrieve latency: {demo_dt*1000:.2f} ms\n")
for h in demo_hits:
    print(f"[{h['score']:.3f}] {h['source']} | {h['chunk_id']}")
    print(h["text"][:220].replace("\n", " "), "...\n")


retrieve latency: 39.79 ms

[0.237] wiki_Business_intelligence.txt | wiki_Business_intelligence.txt::57
ehousing tools (the terms are used so interchangeably that they're often referred to as BI/DW) are extremely expensive [...]   ^ Miller Devens, Richard (1865). Cyclopaedia of Commercial and Business Anecdotes; Comprising ...

[0.159] wiki_Business_intelligence.txt | wiki_Business_intelligence.txt::60
n, Boris (21 November 2008). "Topic Overview: Business Intelligence" .   ^ Evelson, Boris (29 April 2010). "Want to know what Forrester's lead data analysts are thinking about BI and the data domain?" . Archived from the ...

[0.152] wiki_Operations_research.txt | wiki_Operations_research.txt::31
ated the areas were not vital, and adding armor to non-vital areas where damage is acceptable reduces aircraft performance. Their suggestion to remove some of the crew so that an aircraft loss would result in fewer perso ...

[0.146] wiki_Big_data.txt | wiki_Big_data.txt::210
o, and Video?" . Th

## Step 6: Grounded Generation (Ollama)

Retrieved chunks are merged into context and passed to the generator prompt.
Prompt policy:
- answer from context only,
- abstain when evidence is missing,
- provide cited source filenames.

For final evaluated runs, use a real open-weight 7B-14B instruct model.


In [18]:
import requests

RUN_LLM = True  # Set to False if Ollama is unavailable


def build_prompt(question: str, contexts: list[dict]) -> str:
    ctx_blocks = []
    for j, c in enumerate(contexts, start=1):
        ctx_blocks.append(f"[Passage {j} | source: {c['source']}]\n{c['text']}\n")
    ctx = "\n".join(ctx_blocks).strip()

    return f"""You are a careful assistant. Answer the user's question using ONLY the Context below.
If the Context does not contain enough information, say: "I cannot determine this from the provided context." Do not guess.

Context:
{ctx}

User question:
{question}

Style requirements:
- Write in clear English
- Use bullet points if helpful
- End with a single line: "Sources:" listing only filenames that appear in the Context headers
""".strip()


def ollama_generate(prompt: str, model: str = OLLAMA_MODEL, temperature: float = 0.2) -> tuple[str, float]:
    t0 = time.perf_counter()
    resp = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature},
        },
        timeout=600,
    )
    dt = time.perf_counter() - t0
    resp.raise_for_status()
    data = resp.json()
    return data.get("response", "").strip(), dt


def rag_answer(question: str, k: int = TOP_K) -> dict:
    hits, t_ret = retrieve(question, k)
    prompt = build_prompt(question, hits)

    if not RUN_LLM:
        return {
            "question": question,
            "retrieval_ms": t_ret * 1000,
            "generation_ms": None,
            "e2e_ms": t_ret * 1000,
            "hits": hits,
            "answer": "(LLM skipped: set RUN_LLM=True and start Ollama)",
        }

    ans, t_gen = ollama_generate(prompt)
    return {
        "question": question,
        "retrieval_ms": t_ret * 1000,
        "generation_ms": t_gen * 1000,
        "e2e_ms": (t_ret + t_gen) * 1000,
        "hits": hits,
        "answer": ans,
    }


out = rag_answer("List the main stages of a RAG pipeline from documents to an answer.", k=TOP_K)
print(f"retrieval: {out['retrieval_ms']:.2f} ms")
print(f"generation: {out.get('generation_ms')}")
print(f"e2e: {out['e2e_ms']:.2f} ms\n")
print(out["answer"])


retrieval: 72.22 ms
generation: 13913.20899999846
e2e: 13985.43 ms

I cannot determine this from the provided context as there is no information about a RAG pipeline in the given documents. The context only discusses topics related to Business Intelligence, Big Data, Operations Research, and their respective functions.


## Step 7: Retrieval Evaluation

This notebook computes a file-level hit proxy using `data/eval_queries.json`.
For each query, top-k retrieval is counted as a hit if any retrieved source appears in `relevant_sources`.

Use this metric with qualitative grounding analysis in the report.


In [19]:
def precision_at_k_proxy(hits: list[dict], relevant_sources: list[str], k: int) -> float:
    top = hits[:k]
    srcs = {h["source"] for h in top}
    return float(any(s in srcs for s in relevant_sources))


eval_items = json.loads(EVAL_PATH.read_text(encoding="utf-8"))

rows = []
for item in eval_items:
    hits, t_ret = retrieve(item["query"], TOP_K)
    ok = precision_at_k_proxy(hits, item["relevant_sources"], k=TOP_K)
    rows.append(
        {
            "id": item["id"],
            "query": item["query"],
            "hit": ok,
            "top_sources": [h["source"] for h in hits],
            "retrieval_ms": t_ret * 1000,
        }
    )

mean_hit = sum(r["hit"] for r in rows) / len(rows)
print(f"Mean hit rate @TOP_K={TOP_K} (proxy): {mean_hit:.3f}\n")
for r in rows:
    print(r["id"], "HIT" if r["hit"] else "MISS", "| top:", r["top_sources"], "|", f"{r['retrieval_ms']:.1f} ms")


Mean hit rate @TOP_K=4 (proxy): 1.000

q01 HIT | top: ['wiki_Business_analytics.txt', 'wiki_Business_intelligence.txt', 'wiki_Business_intelligence.txt', 'wiki_Business_analytics.txt'] | 270.9 ms
q02 HIT | top: ['wiki_Data_science.txt', 'wiki_Data_science.txt', 'wiki_Data_science.txt', 'wiki_Data_science.txt'] | 13.2 ms
q03 HIT | top: ['wiki_Machine_learning.txt', 'wiki_Machine_learning.txt', 'wiki_Machine_learning.txt', 'wiki_Machine_learning.txt'] | 12.1 ms
q04 HIT | top: ['wiki_Predictive_analytics.txt', 'wiki_Predictive_analytics.txt', 'wiki_Predictive_analytics.txt', 'wiki_Business_analytics.txt'] | 9.9 ms
q05 HIT | top: ['wiki_Big_data.txt', 'wiki_Big_data.txt', 'wiki_Big_data.txt', 'wiki_Big_data.txt'] | 9.7 ms
q06 HIT | top: ['wiki_Business_analytics.txt', 'wiki_Business_intelligence.txt', 'wiki_Business_analytics.txt', 'wiki_Business_intelligence.txt'] | 7.3 ms
q07 HIT | top: ['wiki_Operations_research.txt', 'wiki_Operations_research.txt', 'wiki_Operations_research.txt', 'wiki

## Step 8: Submission Artifacts (Part 1)

Required outputs tied to this notebook:
1. `rag_evaluation_report.md` (metrics, grounding analysis, error attribution, latency)
2. `rag_pipeline_diagram.md` (pipeline architecture)
3. `README.md` (model, serving stack, hardware/runtime, typical latency)

Optional: save a sample trace for reproducibility.


In [9]:
TRACE_DIR = ROOT / "traces"
TRACE_DIR.mkdir(exist_ok=True)

example = rag_answer("Why is a flat FAISS index often a good choice for small corpora?", k=TOP_K)
(TRACE_DIR / "part1_example_trace.json").write_text(
    json.dumps(example, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Wrote:", TRACE_DIR / "part1_example_trace.json")


Wrote: /Users/YUNA/Desktop/ids568-milestone6-660737046/traces/part1_example_trace.json
